# Can we predict late delivery?
Target: `late_delivery`. Prediction is made when the consignment is dispatched, for future consignments, so the split is **temporal**.

**Feature decisions (checklist 7)**
- `consignment`: identifier, dropped (each id appears once, so no entity grouping is needed).
- `claim_paid_eur`: compensation settled *after* delivery, so unknown at dispatch. It is a leak (it is >0 almost only when late), dropped.
- Kept: `carrier`, `distance_km`, `weight_kg`, `parcels`, plus hour and weekday derived from `dispatched_on`.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.base import clone
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, brier_score_loss
df=pd.read_csv('data.csv',parse_dates=['dispatched_on']).sort_values('dispatched_on').reset_index(drop=True)
df['hour']=df.dispatched_on.dt.hour; df['weekday']=df.dispatched_on.dt.weekday
print(df.shape, df.consignment.is_unique)
df.groupby(df.dispatched_on.dt.to_period('M')).late_delivery.agg(['mean','size'])

The late rate changes a lot over time (about 30% early on, about 1% in the last months). This is drift, which matters for how far we can trust any score.

## 1. Test set (checklist 1)
The latest 20% of rows by dispatch time are held out and not touched until the final step.

In [ ]:
cut=int(len(df)*0.8)
train,test=df.iloc[:cut].copy(),df.iloc[cut:].copy()
print(train.dispatched_on.max(), test.dispatched_on.min())
print('late rate train %.3f test %.3f'%(train.late_delivery.mean(),test.late_delivery.mean()))
FEATS=['carrier','distance_km','weight_kg','parcels','hour','weekday']
NUM=['distance_km','weight_kg','parcels','hour','weekday']
Xtr,ytr=train[FEATS],train.late_delivery.values

## 2. Candidate models (checklist 4), named before any scoring
1. `baseline`: predicts the training late rate for everyone.
2. `logreg_C0.1`, `logreg_C1`: logistic regression, one-hot carrier, scaled numerics.
3. `hgb_shallow`, `hgb_deep`: histogram gradient boosting (depth 3 / depth 6, with different learning rates).

Selection metric: mean ROC-AUC across forward-chaining CV folds (inside train only). The decision threshold is the one maximising F1 on pooled out-of-fold predictions of the chosen model.
Preprocessing is inside each pipeline, so it is refit per fold (checklist 3).

In [ ]:
def prep(scale):
    return ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),['carrier']),
                              ('n',StandardScaler() if scale else 'passthrough',NUM)])
cands={
 'baseline':DummyClassifier(strategy='prior'),
 'logreg_C0.1':make_pipeline(prep(True),LogisticRegression(C=0.1,max_iter=1000)),
 'logreg_C1':make_pipeline(prep(True),LogisticRegression(C=1,max_iter=1000)),
 'hgb_shallow':make_pipeline(prep(False),HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=150,random_state=0)),
 'hgb_deep':make_pipeline(prep(False),HistGradientBoostingClassifier(max_depth=6,learning_rate=0.1,max_iter=150,random_state=0)),
}
print(list(cands))

## 3. Forward-chaining cross-validation on train only (checklist 2, 3, 5)

In [ ]:
tscv=TimeSeriesSplit(n_splits=5)
rows=[];oof={}
for name,m in cands.items():
    p=np.full(len(train),np.nan); aucs=[];aps=[]
    for tr,va in tscv.split(Xtr):
        f=clone(m).fit(Xtr.iloc[tr],ytr[tr]); pr=f.predict_proba(Xtr.iloc[va])[:,1]; p[va]=pr
        aucs.append(roc_auc_score(ytr[va],pr)); aps.append(average_precision_score(ytr[va],pr))
    oof[name]=p
    rows.append(dict(model=name,cv_auc=np.mean(aucs),auc_sd=np.std(aucs),cv_ap=np.mean(aps)))
cv=pd.DataFrame(rows).set_index('model').round(4); cv

In [ ]:
# fold prevalences, to read the PR numbers fairly
for i,(tr,va) in enumerate(tscv.split(Xtr)): print(i,len(tr),len(va),round(ytr[va].mean(),3))

## 4. Choose the decision threshold on out-of-fold predictions only
Best model by mean CV AUC: `logreg_C0.1` (ties within noise with `logreg_C1`; the more regularised one is kept).

In [ ]:
best='logreg_C0.1'
ok=~np.isnan(oof[best]); yo,po=ytr[ok],oof[best][ok]
grid=np.linspace(0.05,0.9,86)
f1s=[f1_score(yo,po>=t) for t in grid]
THR=float(grid[int(np.argmax(f1s))])
print('threshold',round(THR,3),'OOF F1 %.3f'%max(f1s),'| F1 of flag-everything %.3f'%f1_score(yo,np.ones_like(yo)))


## 5. Decision, fixed before the test set is scored (checklist 5)
- Model: `logreg_C0.1` (logistic regression, C=0.1, one-hot carrier + scaled numerics), features `carrier, distance_km, weight_kg, parcels, hour, weekday`.
- Threshold: 0.24, from out-of-fold F1 on the training rows.
- Nothing is changed after seeing the test score.

## 6. Refit on all training rows, score the test set once (checklist 6)

In [ ]:
final=clone(cands['logreg_C0.1']).fit(Xtr,ytr)
base=clone(cands['baseline']).fit(Xtr,ytr)
yt=test.late_delivery.values; Xte=test[FEATS]
pt=final.predict_proba(Xte)[:,1]; pb=base.predict_proba(Xte)[:,1]
res=pd.DataFrame({'model':['logreg_C0.1','baseline'],
  'roc_auc':[roc_auc_score(yt,pt),roc_auc_score(yt,pb)],
  'avg_precision':[average_precision_score(yt,pt),average_precision_score(yt,pb)],
  'brier':[brier_score_loss(yt,pt),brier_score_loss(yt,pb)]}).set_index('model').round(4)
print('test n=%d, late=%d (%.3f)'%(len(yt),yt.sum(),yt.mean()))
print('F1 at threshold %.2f: %.3f ; flagged share %.3f ; mean predicted prob %.3f'%(THR,f1_score(yt,pt>=THR),(pt>=THR).mean(),pt.mean()))
res

## Conclusion
See the numbers above. In training CV the features carry moderate signal (AUC about 0.72 against 0.50 for the baseline). The test period differs a lot from training: late rate about 2% against about 26%, so the test has few positives and its estimates are noisy. Probabilities are over-predicted there (calibration drift), and a threshold tuned on earlier data flags far too many consignments. The ranking signal is the part that is usable; any deployment needs recalibration on recent data, and it would be worth finding what changed in late deliveries after October 2025.